In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [98]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model.load("model_5.5M", versionID=5, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/model_5.5M/tokenizer.json
loading the historique from: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/models_save/model_5.5M/versions/version_0005_trained-5epoches/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(256/768) = 1.732051
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=256, window_pattern='SSSL')
5_537_900 total params (embeding: 655_360 | last layer: 163_840 | transformer: 4_718_688)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [99]:
print(f"trained for {model.nb_epoches_done} epoches")
for k, v in model.historique.get_all_historique().items():
    print(f"{k}: {v.get(str(model.nb_epoches_done-1), None):.4g}") # type: ignore

trained for 5 epoches
CE_train: 0.7132
CE2_train: 0.6243
PPL_train: 2.041
PPL2_train: 1.867
BPC_train: 0.5213
ENTROPY_train: 0.7127
LOGITS_SD_train: 3.21
TOP-1_train: 0.8229
TOP-5_train: 0.8683
epoch_duration: 162.6


In [100]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [101]:
N = 75
print(f"using: {dataset.samples[N].svg_file}")
start = dataset.samples[N].txt[: model.context_size]
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
        start=start, decode_batch=256, temperature=1, top_k=5, 
        max_tokens=32000, max_time=2*60, statsPtr=statsPtr):
    results.append(txt)
    print(txt, end="", flush=True)

using: /autofs/unitytravail/travail/jmayol/master 2/pfe/PFE_LLM_art_generation/dataset/samples/0045_circle_packing.svg
7.0002)"><ellipse rx="42.8607" ry="39.0099" style="fill:none;" cx="0" cy="0"/><ellipse rx="38.0808" ry="34.0001" style="fill:none;" cx="0" cy="0"/><ellipse rx="33.0102" ry="30.8697" style="fill:none;" cx="0" cy="0"/><ellipse rx="29.0883" ry="27.0702" style="fill:none;" cx="0" cy="0"/><ellipse rx="25.0811" ry="24.996" style="fill:none;" cx="0" cy="0"/><ellipse rx="21.8602" ry="21.7608" style="fill:none;" cx="0" cy="0"/><ellipse rx="17.0708" ry="18.0775" style="fill:none;" cx="0" cy="0"/><ellipse rx="13.0007" ry="15.0197" style="fill:none;" cx="0" cy="0"/><ellipse rx="9.0199" ry="11.996" style="fill:none;" cx="0" cy="0"/><ellipse rx="4.0883" ry="8.0804" style="fill:none;" cx="0" cy="0"/></g><g style="stroke-linecap:round;" transform="translate(152.0186,259.766) rotate(200.0197)"><ellipse rx="35.7686" ry="33.9702" style="fill:none;" cx="0" cy="0"/><ellipse rx="29.0108" ry

In [109]:
print(start)

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [110]:
results.insert(0, start)

In [111]:
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

GenerationStats(nb_tokens=13517, gen_time=120.00993342900028, stop_reason='reached max_time')
 -> 112.63 tokens/sec


In [112]:
print(results)

['<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:\'Dialog\'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none

In [113]:
print(results[0])

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [114]:
print("".join(results))

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;" transform="translate(431.0456,173.9697) rotate(350.1284)"><ellipse rx="74.4221" ry="48.0768" style="fill:none;" cx="0" cy="0"/><ellipse rx="70.2875" ry="45.4058" style="fill:none;" cx="0" cy="0"/><ellipse rx="66.153" ry="42.7349" style="fill:none;" cx="0" cy="0"/><ellipse rx="62.0184" ry="40.064" style="fill:none;" c

In [108]:
display(SVG(data="".join(results)))

ExpatError: mismatched tag: line 1, column 6283